# Movie Recommendation System - Part 2: Content-Based Filtering

This notebook builds a Content-Based Filtering recommender. We implement two models:
1. **Genre-Based Similarity**: Employs TF-IDF and Cosine Similarity on movie genres.
2. **Tag Genome-Based Similarity**: Computes similarity using the 1128-dimensional tag relevance scores in `genome_scores.csv`.


### Step 1: Import Required Libraries
We import scikit-learn's `TfidfVectorizer` and `cosine_similarity` for feature extraction and similarity calculations.


In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')


### Step 2: Load Sampled Movies and Ratings
We load the sampled movies and check the first few rows.


In [2]:
movies = pd.read_csv('movies_sampled.csv')
ratings = pd.read_csv('ratings_sampled.csv')
print(f'Loaded {len(movies)} movies and {len(ratings)} ratings.')
print(movies.head(3))


Loaded 3159 movies and 2033580 ratings.
   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  


### Step 3: Genre-Based Content Filtering - Data Prep
We replace the '|' separators in the genres column with spaces to create a text representation of genres.


In [3]:
movies['genres_clean'] = movies['genres'].str.replace('|', ' ')
print(movies[['title', 'genres_clean']].head(5))


                                title  \
0                    Toy Story (1995)   
1                      Jumanji (1995)   
2             Grumpier Old Men (1995)   
3            Waiting to Exhale (1995)   
4  Father of the Bride Part II (1995)   

                                  genres_clean  
0  Adventure Animation Children Comedy Fantasy  
1                   Adventure Children Fantasy  
2                               Comedy Romance  
3                         Comedy Drama Romance  
4                                       Comedy  


### Step 4: Compute TF-IDF Matrix for Genres
We convert genres into a numerical vector using TF-IDF representation. Hyphenated terms like 'Sci-Fi' are kept intact.


In [4]:
tfidf = TfidfVectorizer(token_pattern=r'(?u)\b[\w-]+\b')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print('Extracted genre tokens:', tfidf.get_feature_names_out())


TF-IDF matrix shape: (3159, 19)
Extracted genre tokens: ['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'film-noir' 'horror' 'imax' 'musical'
 'mystery' 'romance' 'sci-fi' 'thriller' 'war' 'western']


### Step 5: Compute Cosine Similarity for Genres
Compute pairwise cosine similarity between all movies based on genre TF-IDF vectors.


In [5]:
genre_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f'Genre Similarity Matrix shape: {genre_sim.shape}')


Genre Similarity Matrix shape: (3159, 3159)


### Step 6: Define Genre-Based Recommendation Function
Create a function that looks up a movie index from its title, finds its similarity scores, and outputs the top-N closest movies.


In [6]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def recommend_by_genre(title, top_n=5):
    if title not in indices:
        print(f'Movie "{title}" not found in sampled dataset.')
        return pd.DataFrame()
    
    idx = indices[title]
    sim_scores = list(enumerate(genre_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Filter out input movie and pick top N
    sim_scores = [x for x in sim_scores if x[0] != idx][:top_n]
    
    movie_indices = [x[0] for x in sim_scores]
    scores = [x[1] for x in sim_scores]
    
    recommendations = movies.iloc[movie_indices].copy()
    recommendations['similarity'] = scores
    return recommendations[['title', 'genres', 'similarity']]


### Step 7: Test Genre-Based Recommendations
Verify recommendations for classic movies like 'Toy Story' and 'The Matrix'.


In [7]:
print('Genre-based recommendations for "Toy Story (1995)":')
print(recommend_by_genre('Toy Story (1995)', 5))
print('\nGenre-based recommendations for "Matrix, The (1999)":')
print(recommend_by_genre('Matrix, The (1999)', 5))


Genre-based recommendations for "Toy Story (1995)":
                                               title  \
1236                                     Antz (1998)   
1655                              Toy Story 2 (1999)   
1904  Adventures of Rocky and Bullwinkle, The (2000)   
1997                Emperor's New Groove, The (2000)   
2192                           Monsters, Inc. (2001)   

                                           genres  similarity  
1236  Adventure|Animation|Children|Comedy|Fantasy         1.0  
1655  Adventure|Animation|Children|Comedy|Fantasy         1.0  
1904  Adventure|Animation|Children|Comedy|Fantasy         1.0  
1997  Adventure|Animation|Children|Comedy|Fantasy         1.0  
2192  Adventure|Animation|Children|Comedy|Fantasy         1.0  

Genre-based recommendations for "Matrix, The (1999)":
                                         title                  genres  \
54   Lawnmower Man 2: Beyond Cyberspace (1996)  Action|Sci-Fi|Thriller   
62                      

### Step 8: Tag Genome-Based Content Filtering - Load and Filter
The MovieLens tag genome represents how strongly different tags apply to movies. We load the raw `genome_scores.csv` and filter it for our sampled movies.


In [8]:
genome_scores = pd.read_csv('genome_scores.csv')
sampled_movie_ids = set(movies['movieId'])
genome_scores_filtered = genome_scores[genome_scores['movieId'].isin(sampled_movie_ids)]
print(f'Filtered genome scores row count: {len(genome_scores_filtered)}')


Filtered genome scores row count: 3559968


### Step 9: Pivot Genome Scores to Movie-Tag Matrix
We transform the data into a movie-by-tag matrix where columns are the 1128 unique tags.


In [9]:
genome_pivot = genome_scores_filtered.pivot(index='movieId', columns='tagId', values='relevance').fillna(0)
print(f'Genome pivot matrix shape (movies x tags): {genome_pivot.shape}')


Genome pivot matrix shape (movies x tags): (3156, 1128)


### Step 10: Compute Cosine Similarity for Genome Vectors
Calculate movie similarities based on high-dimensional tag relevance.


In [10]:
genome_sim = cosine_similarity(genome_pivot, genome_pivot)
print(f'Genome Similarity Matrix shape: {genome_sim.shape}')

genome_movie_ids = genome_pivot.index.tolist()
movie_id_to_genome_idx = {movie_id: idx for idx, movie_id in enumerate(genome_movie_ids)}


Genome Similarity Matrix shape: (3156, 3156)


### Step 11: Define Genome-Based Recommendation Function
Define the recommendation logic using genome tags cosine similarity.


In [11]:
def recommend_by_genome(title, top_n=5):
    movie_row = movies[movies['title'] == title]
    if movie_row.empty:
        print(f'Movie "{title}" not found in sampled dataset.')
        return pd.DataFrame()
    
    movie_id = movie_row['movieId'].values[0]
    if movie_id not in movie_id_to_genome_idx:
        print(f'Movie "{title}" does not have genome relevance tags.')
        return pd.DataFrame()
    
    idx = movie_id_to_genome_idx[movie_id]
    sim_scores = list(enumerate(genome_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Filter out input movie
    sim_scores = [x for x in sim_scores if genome_movie_ids[x[0]] != movie_id][:top_n]
    
    rec_movie_ids = [genome_movie_ids[x[0]] for x in sim_scores]
    scores = [x[1] for x in sim_scores]
    
    recommendations = movies[movies['movieId'].isin(rec_movie_ids)].copy()
    recommendations['similarity'] = recommendations['movieId'].map(dict(zip(rec_movie_ids, scores)))
    recommendations = recommendations.sort_values(by='similarity', ascending=False)
    return recommendations[['title', 'genres', 'similarity']]


### Step 12: Test Genome-Based Recommendations
Verify recommendations using tag genome similarity for 'Toy Story' and 'The Matrix'.


In [12]:
print('Genome-based recommendations for "Toy Story (1995)":')
print(recommend_by_genome('Toy Story (1995)', 5))
print('\nGenome-based recommendations for "Matrix, The (1999)":')
print(recommend_by_genome('Matrix, The (1999)', 5))


Genome-based recommendations for "Toy Story (1995)":
                      title                                            genres  \
2192  Monsters, Inc. (2001)       Adventure|Animation|Children|Comedy|Fantasy   
1655     Toy Story 2 (1999)       Adventure|Animation|Children|Comedy|Fantasy   
1272   Bug's Life, A (1998)               Adventure|Animation|Children|Comedy   
2414    Finding Nemo (2003)               Adventure|Animation|Children|Comedy   
3061     Toy Story 3 (2010)  Adventure|Animation|Children|Comedy|Fantasy|IMAX   

      similarity  
2192    0.958831  
1655    0.955522  
1272    0.948892  
2414    0.924944  
3061    0.920019  

Genome-based recommendations for "Matrix, The (1999)":
                            title  \
2351           Equilibrium (2002)   
3064             Inception (2010)   
2410  Matrix Reloaded, The (2003)   
3156      Edge of Tomorrow (2014)   
2294       Minority Report (2002)   

                                               genres  similarity  

### Step 13: Save Genome Similarity Matrices
We save these similarity results to files for use in the hybrid model in Notebook 4.


In [13]:
np.save('genome_similarity.npy', genome_sim)
np.save('genome_movie_ids.npy', np.array(genome_movie_ids))
print('Saved "genome_similarity.npy" and "genome_movie_ids.npy" to disk.')


Saved "genome_similarity.npy" and "genome_movie_ids.npy" to disk.
